# Codificación de Huffman

Compresión sin pérdida que asigna códigos binarios más cortos a los símbolos más frecuentes. Se construye un árbol uniendo repetidamente los dos nodos de menor frecuencia (con una cola de prioridad); el camino hasta cada hoja (0 = izquierda, 1 = derecha) es su código. Ningún código es prefijo de otro, así que la decodificación no es ambigua.

## 1. Tabla de códigos

Construir el árbol de Huffman de un texto con `heapq` y obtener el código de cada carácter.

In [1]:
import heapq
from collections import Counter
from itertools import count

def codigos_huffman(texto):
    desempate = count()
    # (frecuencia, desempate, árbol); hoja = carácter, nodo = (izq, der)
    cola = [(f, next(desempate), c) for c, f in Counter(texto).items()]
    heapq.heapify(cola)
    while len(cola) > 1:
        f1, _, a = heapq.heappop(cola)
        f2, _, b = heapq.heappop(cola)
        heapq.heappush(cola, (f1 + f2, next(desempate), (a, b)))

    codigos = {}
    def recorre(nodo, codigo):
        if isinstance(nodo, str):
            codigos[nodo] = codigo or "0"
        else:
            recorre(nodo[0], codigo + "0")
            recorre(nodo[1], codigo + "1")
    recorre(cola[0][2], "")
    return codigos

texto = "abracadabra"
codigos = codigos_huffman(texto)
print(codigos)

{'a': '0', 'c': '100', 'd': '101', 'b': '110', 'r': '111'}


## 2. Codificar y decodificar

Codificar un texto con la tabla, decodificarlo leyendo bit a bit y comparar el tamaño con 8 bits por carácter.

In [2]:
def codifica(texto, codigos):
    return "".join(codigos[c] for c in texto)

def decodifica(bits, codigos):
    inverso = {v: k for k, v in codigos.items()}
    resultado, actual = [], ""
    for bit in bits:
        actual += bit
        if actual in inverso:
            resultado.append(inverso[actual])
            actual = ""
    return "".join(resultado)

texto = "en un lugar de la mancha de cuyo nombre no quiero acordarme"
codigos = codigos_huffman(texto)
bits = codifica(texto, codigos)
assert decodifica(bits, codigos) == texto
print(f"{len(texto) * 8} bits -> {len(bits)} bits ({len(bits) / (len(texto) * 8):.0%})")

472 bits -> 222 bits (47%)
